# Part 3 probe: does ensembling help at all?

Before any test-like set or teammate models, this notebook answers one question: **does averaging the O/B/I probabilities of several BiLSTMs beat the best single one?**

* **Members** (all trained on the same 100k `fit` rows, early-stopped on `checkpoint`, same frozen split):
  * `word_s42`: the frozen Part 2 model (Kaggle public 0.46103), loaded, not retrained.
  * `word_s43`, `word_s44`: the same configuration with other seeds (pure variance reduction).
  * `gru2`: 2-layer BiGRU instead of BiLSTM (a different cell).
  * `bpe`: the Part 2 BPE tokenizer, projected back onto words with the first-piece convention (different inputs).
* **Splits:**
  * `calibration` (2k) is used by no member, so any choice of subset or weights is made on it.
  * `checkpoint` (4k) is the honest report. It was used for early stopping by every member, which inflates all of them about equally.
  * `labeling` (26 organiser examples) is only indicative.
* **No tuning first:** section 4 compares members and averages at Part 2's fixed threshold 0.8 with the strict filter.
* **Uncertainty:** a paired bootstrap over documents gives a 95% interval for ensemble − best single.
* **One Kaggle check:** the all-member average at 0.8/strict, compared with 0.46103 at the same decoding settings.

Training is resumable: results go to `ROOT/results/ensemble_probe/` and finished members are reused.

## 0. Setup (GPU runtime)

In [ ]:
# Mount Drive from the Colab sidebar first (Files -> Mount Drive); this notebook never mounts it.
from pathlib import Path
import json, os, shutil, subprocess, sys

if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Google Drive is not mounted: left sidebar -> folder icon -> "Mount Drive", then rerun this cell')
CANDIDATE_ROOTS = [os.environ.get('A3_ROOT', ''), '/content/drive/MyDrive/KSE_AI301_A3',
                   '/content/drive/.shortcut-targets-by-id/12ao27pY1xBfLeIRZcJYTDFojxW5-88tc/KSE_AI301_A3']
ROOT = next((Path(p) for p in CANDIDATE_ROOTS if p and Path(p).is_dir()), None)
if ROOT is None:
    raise RuntimeError('KSE_AI301_A3 not found. Checked:\n'+'\n'.join(p for p in CANDIDATE_ROOTS if p)
                       +'\nAdd a shortcut to the shared folder in My Drive, or set A3_ROOT to its path')
WORK = Path('/content/a3'); WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK/'A3'                      # fresh clone of the committed code, never a hand-copied file
BRANCH = 'ensemble'
if not (PROJECT/'.git').exists():
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/Taburetochka60/A3.git', str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'pull', '-q', '--ff-only'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip()
sys.path.insert(0, str(PROJECT)); os.chdir(PROJECT)
os.environ['MPLCONFIGDIR'] = '/tmp/a3_matplotlib'

import importlib.metadata, torch
version = importlib.metadata.version('torch')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements-part2.txt')])
assert importlib.metadata.version('torch') == version

from part2_data import file_hash
from part2_compat import pin_grouping_hash
pin_grouping_hash()                      # Colab's Python 3.13 changed ast.dump; keeps the split's code-hash guard valid
SPLIT = PROJECT/'part2_split.json'
ENSEMBLE_SPLIT = PROJECT/'ensemble_split.json'
spec = json.loads(SPLIT.read_text())
DATA = WORK/'data'; DATA.mkdir(exist_ok=True)
DATA_SOURCES = [ROOT/'data', ROOT, ROOT/'domain_shift_source'/'data']
for name, expected in spec['files'].items():      # local copy: much faster to read than Drive
    if not (DATA/name).exists() or file_hash(DATA/name) != expected:
        source = next((d/name for d in DATA_SOURCES if (d/name).is_file()), None)
        if source is None:
            raise RuntimeError(f'{name} not found in any of: {[str(d) for d in DATA_SOURCES]}')
        shutil.copy2(source, DATA/name)
    if file_hash(DATA/name) != expected:
        raise RuntimeError(f'{name} differs from the file part2_split.json was built from')

# The frozen Part 2 checkpoint, found by its recorded hash.
REFERENCE_CANDIDATES = [ROOT/'part2_model'/'checkpoint.pt',
                        ROOT/'results'/'clean_final'/'runs'/'CLEAN_LSTM2_100k_seed42_f362fa37df'/'best.pt']
REFERENCE = next((p for p in REFERENCE_CANDIDATES
                  if p.exists() and file_hash(p) == spec['reference']['checkpoint_sha256']), None)
if REFERENCE is None:
    raise RuntimeError('Frozen Part 2 checkpoint not found under ROOT; add its path to REFERENCE_CANDIDATES')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('root:', ROOT, '\ncode:', BRANCH, COMMIT, '| python', sys.version.split()[0], '| device', DEVICE,
      '\nreference checkpoint:', REFERENCE)

In [ ]:
# W&B is optional: training logs automatically when a WANDB_API_KEY Colab secret exists.
# Runs go to your own W&B account unless a WANDB_ENTITY secret names a team you belong to;
# an entity your key cannot write to fails with "403 ... cannot access this resource".
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if secret('WANDB_API_KEY'):
    os.environ['WANDB_API_KEY'] = secret('WANDB_API_KEY')
    os.environ.setdefault('WANDB_PROJECT', 'ai301-a3')
    if secret('WANDB_ENTITY'):
        os.environ['WANDB_ENTITY'] = secret('WANDB_ENTITY')
    else:
        os.environ.pop('WANDB_ENTITY', None)
    import wandb
    entity = os.environ.get('WANDB_ENTITY') or wandb.Api().default_entity
    print('W&B logging: enabled ->', f"{entity}/{os.environ['WANDB_PROJECT']}")
else:
    print('W&B logging: disabled')

## 1. Data

In [ ]:
from part2_data import load_data, prepare_data
from ensemble_data import load_docs
frames, spec = load_data(DATA, SPLIT)
data = prepare_data(frames, spec)                 # fit + checkpoint documents for training
SPLITS = ['calibration', 'checkpoint', 'labeling']
docs = load_docs(DATA, SPLITS+['test'], SPLIT)    # reference documents every bundle is aligned to
print({k: len(v) for k, v in docs.items()})

## 2. Members

New members reuse `part2.train` through `train_member`, which only works around a key clash in its epoch record. The configuration differs from the frozen model only in the listed keys. Each one is logged to W&B when it is enabled. Expect a few minutes per member on a T4. Rerunning the cell skips finished members.

In [ ]:
from part2 import configuration, gpu
from ensemble_members import train_member      # part2.train plus a fix for its 'precision' key clash
RESULTS = ROOT/'results'/'ensemble_probe'
BUNDLES = RESULTS/'bundles'
VARIANTS = {
    'word_s43': dict(seed=43),
    'word_s44': dict(seed=44),
    'gru2': dict(rnn_type='gru'),
    'bpe': dict(tokenizer='bpe'),
}
MEMBERS = [dict(name='word_s42', kind='part2', checkpoint=REFERENCE,
                expected=dict(split='calibration', threshold=.8, policy='strict',
                              f1=spec['reference']['metrics']['calibration']['f1']))]
print(gpu())
for name, changes in VARIANTS.items():
    result = train_member(configuration(name='probe_'+name, **changes), data['fit'], data['checkpoint'], spec,
                   RESULTS/'members'/name, WORK/'runs')
    # train() scores checkpoint at threshold 0 with the original filter; the bundle must reproduce it.
    MEMBERS.append(dict(name=name, kind='part2', checkpoint=Path(result['checkpoint']), wandb_run=result['identity'][:8],
                        expected=dict(split='checkpoint', threshold=0., policy='original', f1=result['metrics']['f1'])))
    print(name, 'best epoch', result['best_epoch'], 'checkpoint F1', round(result['metrics']['f1'], 4))

## 3. Probability bundles (one pass per member and split; every member must reproduce its logged F1)

In [ ]:
from ensemble_members import export_member
for member in MEMBERS:
    print(member['name'], 'verified F1:', export_member(member, docs, BUNDLES, DEVICE))

## 4. Does averaging help? Fixed threshold 0.8, strict filter, no tuning

Every subset of members is averaged with equal weights. The table lists the single members, the all-member average and the subset that is best on calibration.

In [ ]:
import numpy as np, pandas as pd
from ensemble import blend, bootstrap_delta, document_counts, load_members, subset_search
names = [m['name'] for m in MEMBERS]
P = {s: load_members(BUNDLES, names, s, docs[s]) for s in SPLITS}
fixed = pd.concat([subset_search(P[s], docs[s], thresholds=(.8,), max_size=len(names)).assign(split=s) for s in SPLITS])
fixed = fixed.pivot_table(index=['members', 'size'], columns='split', values='f1').reset_index().set_index('members')
singles = fixed[fixed['size'].eq(1)].sort_values('calibration', ascending=False)
BEST_SINGLE = singles.index[0]
EVERYTHING = '+'.join(sorted(names))
SEEDS = '+'.join(sorted(n for n in names if n.startswith('word_')))
BEST_COMBO = fixed.sort_values(['calibration', 'size'], ascending=[False, True], kind='stable').index[0]
shown = list(dict.fromkeys(singles.index.tolist()+[SEEDS, EVERYTHING, BEST_COMBO]))
print('best single on calibration:', BEST_SINGLE, '| best subset on calibration:', BEST_COMBO)
fixed.loc[shown, ['size']+SPLITS].round(4)

**Paired bootstrap** (2,000 resamples of documents): F1(ensemble) − F1(best single), with a 95% interval. `BEST_COMBO` was picked on calibration, so its calibration row is optimistic, and only its checkpoint row is a fair test.

In [ ]:
rows = []
for combo in dict.fromkeys([SEEDS, EVERYTHING, BEST_COMBO]):
    if '+' not in combo:                           # compare real ensembles only
        continue
    for s in ['calibration', 'checkpoint']:
        a = document_counts(docs[s], P[s][BEST_SINGLE], .8)
        b = document_counts(docs[s], blend(P[s], dict.fromkeys(combo.split('+'), 1.)), .8)
        rows.append(dict(ensemble=combo, versus=BEST_SINGLE, split=s, **bootstrap_delta(a, b)))
pd.DataFrame(rows).round(4)

Diversity: the share of checkpoint documents where two members output different location sets. If members almost never disagree, averaging cannot change much.

In [ ]:
from part2 import decode
predicted = {n: [set(l) for l in decode(docs['checkpoint'], P['checkpoint'][n], .8, 'strict')[0]] for n in names}
pd.DataFrame([[np.mean([x != y for x, y in zip(predicted[a], predicted[b])]) for b in names] for a in names],
             index=names, columns=names).round(3)

## 5. The real pipeline: tuned recipe

`run_search` tries greedy weighted averaging, every subset of up to 4 members and every single member over thresholds 0–0.9. It selects on **calibration** (the test-like set does not exist yet) and reports on checkpoint and labeling.

In [ ]:
from ensemble import log_wandb, run_search
SEARCH = RESULTS/'search_calibration_v1'          # new folder for every change of members or settings
result, report = run_search(BUNDLES, names, docs, SEARCH, select='calibration', report=('checkpoint', 'labeling'))
log_wandb('probe_'+SEARCH.name, SEARCH)
print('final:', result['final'])
report.pivot_table(index='recipe', columns='split', values='f1').sort_values('calibration', ascending=False).round(4)

## 6. One Kaggle check

The all-member average uses the same decoding as the 0.46103 submission (threshold 0.8, strict filter), so the difference comes from averaging alone. Submit it by hand with the printed command or through the Kaggle website (limit: 10 per day), then record the real score.

In [ ]:
from ensemble import export_test, recipe
PROBE = recipe('probe_all_average', dict.fromkeys(names, 1.), .8, 'strict', 'none (Part 2 fixed decoding)', None)
submission = export_test(PROBE, BUNDLES, docs['test'], RESULTS/'submissions'/'all_average_t080_strict.csv')
print(f'kaggle competitions submit -c kse-ua-location-extraction-2026 -f "{submission}" -m "probe all-average t0.8 strict ({COMMIT})"')

In [ ]:
# Only after the score is visible on Kaggle; never enter a guessed number.
# from ensemble import record_submission
# record_submission(RESULTS/'leaderboard.csv', PROBE, submission, public_f1=..., note='probe: all members averaged, t=0.8 strict')

## How to read the result

* **Ensembling helps (in-distribution)** if the all-member average beats the best single member on checkpoint and the bootstrap interval lies above 0. A typical gain from averaging similar models is about +0.005 to +0.02 F1.
* **Seeds only vs. everything:** if the `word_*` average gains as much as the full ensemble, the gain is variance reduction. If GRU or BPE add more, input and architecture diversity matter, which favours adding the n-gram models later.
* **Kaggle:** the public test has only 477 texts, so differences of about ±0.01 from 0.46103 are within noise. A clear jump would mean averaging also helps under the domain shift. No change would mean the gap is not about variance, and the test-like threshold and OOV-robust members matter more.
* Record the numbers (both tables and the Kaggle score) for the report's experiment table.